# Project test
---
## Google Gemini prompt logs
Located below code


In [7]:
import time
import csv
import math
import tkinter as tk
from tkinter import ttk, messagebox, filedialog
import matplotlib
matplotlib.use("TkAgg")
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from matplotlib.figure import Figure

class FlashDrumSimulation:
    """Backend dynamic flash drum model and PID controllers."""
    def __init__(self):
        # Inlet variables
        self.F_in = 15.0       # mol/s
        self.x_butanol = 0.5   # mole fraction
        self.T_in = 400.0      # K

        # State variables
        self.T = 453.0         # Temperature (K)
        self.P = 69.8          # Pressure (kPa)
        self.Lvl = 18.7        # Liquid Level (%)
        self.Q = 500.0         # Heat duty (kW)
        self.lift = 0.5        # Vapor valve lift (0-1)
        self.v_out = 0.66      # Bottoms liquid flow rate (L/s)

        # Controllers Setup (Auto / Manual modes)
        # Pressure Controller (PC) -> manipulates Lift
        self.pc_mode = "Auto"
        self.pc_stpt = 74.6
        self.pc_Kc = 0.01
        self.pc_tauI = 36000.0
        self.pc_err_integral = 0.0

        # Temperature Controller (TC) -> manipulates Q
        self.tc_mode = "Auto"
        self.tc_stpt = 455.0
        self.tc_Kc = 10.0
        self.tc_tauI = 36000.0
        self.tc_err_integral = 0.0

        # Level Controller (LC) -> manipulates v_out
        self.lc_mode = "Auto"
        self.lc_stpt = 17.6
        self.lc_Kc = 0.05
        self.lc_tauI = 36000.0
        self.lc_err_integral = 0.0

        # Data History for Plots
        self.time_history = []
        self.T_history = []
        self.Q_history = []
        self.Lvl_history = []
        self.v_out_history = []
        self.P_history = []
        self.lift_history = []

        self.current_time = 0.0

    def step(self, dt):
        """Advances the differential equations based on real elapsed time delta dt."""
        if dt <= 0:
            return

        # 1. Update PID Controllers
        # PC
        if self.pc_mode == "Auto":
            err_P = self.P - self.pc_stpt
            self.pc_err_integral += err_P * dt
            i_term = (self.pc_Kc / self.pc_tauI) * self.pc_err_integral if self.pc_tauI > 0 else 0
            self.lift = max(0.0, min(1.0, 0.5 + self.pc_Kc * err_P + i_term))

        # TC
        if self.tc_mode == "Auto":
            err_T = self.tc_stpt - self.T
            self.tc_err_integral += err_T * dt
            i_term = (self.tc_Kc / self.tc_tauI) * self.tc_err_integral if self.tc_tauI > 0 else 0
            self.Q = max(0.0, min(2000.0, 500.0 + self.tc_Kc * err_T + i_term))

        # LC
        if self.lc_mode == "Auto":
            err_L = self.Lvl - self.lc_stpt
            self.lc_err_integral += err_L * dt
            i_term = (self.lc_Kc / self.lc_tauI) * self.lc_err_integral if self.lc_tauI > 0 else 0
            self.v_out = max(0.0, min(10.0, 0.66 + self.lc_Kc * err_L + i_term))

        # 2. Physics & Dynamic Balance Approximations
        dT_dt = (self.Q - (self.T - self.T_in) * 2.0 - (self.T - 400.0) * 0.5) / 50.0
        self.T += dT_dt * dt

        vapor_gen = max(0.0, (self.T - 420.0) * 1.5)
        vapor_out = self.lift * (self.P / 10.0) * 2.0
        dP_dt = (vapor_gen - vapor_out) / 2.0
        self.P = max(10.0, self.P + dP_dt * dt)

        dLvl_dt = (self.F_in * (1.0 - self.x_butanol) * 0.1) - (self.v_out * 0.2)
        self.Lvl = max(0.0, min(100.0, self.Lvl + dLvl_dt * dt))

        # 3. Record History
        self.current_time += dt
        self.time_history.append(self.current_time)
        self.T_history.append(self.T)
        self.Q_history.append(self.Q)
        self.Lvl_history.append(self.Lvl)
        self.v_out_history.append(self.v_out)
        self.P_history.append(self.P)
        self.lift_history.append(self.lift)

        # Limit memory window (last 300 points)
        if len(self.time_history) > 300:
            self.time_history.pop(0)
            self.T_history.pop(0)
            self.Q_history.pop(0)
            self.Lvl_history.pop(0)
            self.v_out_history.pop(0)
            self.P_history.pop(0)
            self.lift_history.pop(0)


class FlashDrumGUI:
    def __init__(self, root):
        self.root = root
        self.root.title("Aniline-Butanol Flash Drum Simulator")
        self.root.geometry("1100x850")
        self.root.configure(bg="#e8e8e8")

        self.sim = FlashDrumSimulation()
        self.sim_speed = 2.0
        self.last_update_time = time.perf_counter()

        self.setup_ui()
        self.setup_plots()
        self.update_loop()

    def setup_ui(self):
        # Top Header Frame
        top_frame = tk.Frame(self.root, bg="#e8e8e8")
        top_frame.pack(side=tk.TOP, fill=tk.X, padx=10, pady=5)

        # Inlet Controls Frame
        inlet_box = tk.LabelFrame(top_frame, text="inlet conditions", font=("Arial", 11, "bold"), bg="white")
        inlet_box.pack(side=tk.LEFT, padx=10)

        tk.Button(inlet_box, text="Update", bg="#007bff", fg="white", font=("Arial", 9, "bold"),
                  command=self.update_inlets).grid(row=0, column=0, columnspan=2, pady=2, sticky="ew")

        tk.Label(inlet_box, text="F_in (mol/s)", bg="white").grid(row=1, column=0, sticky="w", padx=5)
        self.entry_Fin = tk.Entry(inlet_box, width=8)
        self.entry_Fin.insert(0, "15")
        self.entry_Fin.grid(row=1, column=1, padx=5, pady=2)

        tk.Label(inlet_box, text="x_butanol", bg="white").grid(row=2, column=0, sticky="w", padx=5)
        self.entry_xbut = tk.Entry(inlet_box, width=8)
        self.entry_xbut.insert(0, "0.5")
        self.entry_xbut.grid(row=2, column=1, padx=5, pady=2)

        tk.Label(inlet_box, text="T_in (K)", bg="white").grid(row=3, column=0, sticky="w", padx=5)
        self.entry_Tin = tk.Entry(inlet_box, width=8)
        self.entry_Tin.insert(0, "400")
        self.entry_Tin.grid(row=3, column=1, padx=5, pady=2)

        # Speed, Snapshot, Directions
        ctrl_box = tk.Frame(top_frame, bg="#e8e8e8")
        ctrl_box.pack(side=tk.LEFT, padx=20)

        speed_frame = tk.Frame(ctrl_box, bg="#d0e0e3", bd=1, relief="solid")
        speed_frame.pack(pady=2, fill=tk.X)
        self.speed_lbl = tk.Label(speed_frame, text=f"simulation speed: {int(self.sim_speed)}", bg="#d0e0e3", font=("Arial", 10))
        self.speed_lbl.pack()
        self.speed_slider = ttk.Scale(speed_frame, from_=1, to=10, value=2, command=self.change_speed)
        self.speed_slider.pack(padx=5, pady=2)

        tk.Button(ctrl_box, text="Snapshot .CSV", bg="#28a745", fg="white", font=("Arial", 10, "bold"),
                  command=self.save_csv).pack(pady=4, fill=tk.X)
        tk.Button(ctrl_box, text="Directions", bg="#5a6268", fg="white", font=("Arial", 10, "bold"),
                  command=self.show_directions).pack(pady=2, fill=tk.X)

        # Diagram & Controller Box Area
        center_frame = tk.Frame(self.root, bg="#e8e8e8")
        center_frame.pack(side=tk.TOP, fill=tk.BOTH, expand=True, padx=10, pady=5)

        # Left Controllers (TC)
        left_ctrl_frame = tk.Frame(center_frame, bg="#e8e8e8")
        left_ctrl_frame.pack(side=tk.LEFT, anchor="n", padx=10)

        self.tc_box = self.create_controller_ui(left_ctrl_frame, "temperature controller", "Q (kW)", "T stpt (K)",
                                                 "500", "455", "0", "36000", self.update_tc)

        # Center PFD Schematic (Canvas)
        self.canvas = tk.Canvas(center_frame, width=320, height=260, bg="#e8e8e8", highlightthickness=0)
        self.canvas.pack(side=tk.LEFT, expand=True)
        self.draw_pfd()

        # Right Controllers (PC and LC)
        right_ctrl_frame = tk.Frame(center_frame, bg="#e8e8e8")
        right_ctrl_frame.pack(side=tk.RIGHT, anchor="n", padx=10)

        self.pc_box = self.create_controller_ui(right_ctrl_frame, "pressure controller", "Lift", "P stpt (kPa)",
                                                 "0.5", "74.6", "0", "36000", self.update_pc)
        self.lc_box = self.create_controller_ui(right_ctrl_frame, "level controller", "v_out (L/s)", "Lvl stpt (%)",
                                                 "0.66", "17.6", "0", "36000", self.update_lc)

    def create_controller_ui(self, parent, title, mv_name, stpt_name, mv_val, stpt_val, kc_val, tau_val, update_cmd):
        box = tk.LabelFrame(parent, text=title, font=("Arial", 11, "bold"), bg="white")
        box.pack(pady=5, fill=tk.X)

        mode_frame = tk.Frame(box, bg="white")
        mode_frame.grid(row=0, column=0, columnspan=2, pady=2, sticky="ew")

        mode_var = tk.StringVar(value="Auto")
        auto_btn = tk.Radiobutton(mode_frame, text="Auto", value="Auto", variable=mode_var, indicatoron=False, width=6, bg="#6c757d", selectcolor="#007bff", fg="white")
        man_btn = tk.Radiobutton(mode_frame, text="Manual", value="Manual", variable=mode_var, indicatoron=False, width=6, bg="#6c757d", selectcolor="#007bff", fg="white")
        auto_btn.pack(side=tk.LEFT, padx=2)
        man_btn.pack(side=tk.LEFT, padx=2)

        tk.Button(mode_frame, text="Update", bg="#007bff", fg="white", font=("Arial", 8, "bold"), command=update_cmd).pack(side=tk.RIGHT, padx=2)

        labels = [mv_name, stpt_name, "Kc", "taul (s)"]
        defaults = [mv_val, stpt_val, kc_val, tau_val]
        entries = {}

        for idx, (lbl, df) in enumerate(zip(labels, defaults), start=1):
            tk.Label(box, text=lbl, bg="white", fg="blue" if idx==1 else ("red" if idx==2 else "black")).grid(row=idx, column=0, sticky="w", padx=5)
            e = tk.Entry(box, width=8)
            e.insert(0, df)
            e.grid(row=idx, column=1, padx=5, pady=2)
            entries[lbl] = e

        box.mode_var = mode_var
        box.entries = entries
        return box

    def setup_plots(self):
        plot_frame = tk.Frame(self.root, bg="white")
        plot_frame.pack(side=tk.BOTTOM, fill=tk.BOTH, expand=True, padx=10, pady=10)

        self.fig = Figure(figsize=(10, 3.0), dpi=100)
        
        # Subplot 1: Temp & Heat Duty
        self.ax1 = self.fig.add_subplot(131)
        self.ax1_twin = self.ax1.twinx()
        self.line_T, = self.ax1.plot([], [], 'g-', label="Temp (K)")
        self.line_Q, = self.ax1_twin.plot([], [], 'b-', label="Q (kW)")
        self.ax1.set_title("temperature and heat duty", fontsize=9)
        self.ax1.set_xlabel("time (s)", fontsize=8)
        self.ax1.set_ylabel("temperature (K)", color='g', fontsize=8)
        self.ax1_twin.set_ylabel("heat duty (kW)", color='b', fontsize=8)

        # Subplot 2: Level & Flow
        self.ax2 = self.fig.add_subplot(132)
        self.ax2_twin = self.ax2.twinx()
        self.line_Lvl, = self.ax2.plot([], [], 'g-', label="Level (%)")
        self.line_vout, = self.ax2_twin.plot([], [], 'b-', label="Flow (L/s)")
        self.ax2.set_title("level and bottoms flow rate", fontsize=9)
        self.ax2.set_xlabel("time (s)", fontsize=8)
        self.ax2.set_ylabel("level (%)", color='g', fontsize=8)
        self.ax2_twin.set_ylabel("bottoms flow (L / s)", color='b', fontsize=8)

        # Subplot 3: Pressure & Valve Lift
        self.ax3 = self.fig.add_subplot(133)
        self.ax3_twin = self.ax3.twinx()
        self.line_P, = self.ax3.plot([], [], 'g-', label="Pressure (kPa)")
        self.line_lift, = self.ax3_twin.plot([], [], 'b-', label="Lift")
        self.ax3.set_title("pressure and valve lift", fontsize=9)
        self.ax3.set_xlabel("time (s)", fontsize=8)
        self.ax3.set_ylabel("pressure (kPa)", color='g', fontsize=8)
        self.ax3_twin.set_ylabel("valve lift", color='b', fontsize=8)

        self.fig.tight_layout(pad=2.0)
        self.canvas_plot = FigureCanvasTkAgg(self.fig, master=plot_frame)
        self.canvas_plot.get_tk_widget().pack(fill=tk.BOTH, expand=True)

    def draw_pfd(self):
        self.canvas.delete("all")
        # Drum body
        self.canvas.create_rectangle(120, 50, 200, 200, fill="#d3d3d3", outline="black", width=2)
        self.canvas.create_arc(120, 30, 200, 70, start=0, extent=180, fill="#d3d3d3", outline="black", width=2)
        self.canvas.create_arc(120, 180, 200, 220, start=180, extent=180, fill="#d3d3d3", outline="black", width=2)

        # Liquid level in drum
        lvl_h = max(5, int((self.sim.Lvl / 100.0) * 120))
        self.canvas.create_rectangle(121, 200 - lvl_h, 199, 200, fill="#d2b48c", outline="")

        # Label on drum
        self.canvas.create_text(160, 90, text="aniline\nbutanol", font=("Arial", 10, "bold"))

        # Controller badges & dynamic numbers
        self.canvas.create_oval(230, 30, 270, 70, fill="white", outline="black", width=2)
        self.canvas.create_text(250, 50, text="PC", font=("Arial", 11, "bold"))
        self.canvas.create_text(250, 18, text=f"{self.sim.P:.1f} kPa", fill="green", font=("Arial", 10, "bold"))

        self.canvas.create_oval(230, 140, 270, 180, fill="white", outline="black", width=2)
        self.canvas.create_text(250, 160, text="LC", font=("Arial", 11, "bold"))
        self.canvas.create_text(250, 122, text=f"{self.sim.Lvl:.1f} %", fill="green", font=("Arial", 10, "bold"))

        self.canvas.create_oval(50, 140, 90, 180, fill="white", outline="black", width=2)
        self.canvas.create_text(70, 160, text="TC", font=("Arial", 11, "bold"))
        self.canvas.create_text(70, 198, text=f"{self.sim.T:.0f} K", fill="green", font=("Arial", 10, "bold"))

    def update_inlets(self):
        try:
            self.sim.F_in = float(self.entry_Fin.get())
            self.sim.x_butanol = float(self.entry_xbut.get())
            self.sim.T_in = float(self.entry_Tin.get())
        except ValueError:
            messagebox.showerror("Error", "Invalid numeric inlet conditions")

    def update_pc(self):
        try:
            self.sim.pc_mode = self.pc_box.mode_var.get()
            self.sim.lift = float(self.pc_box.entries["Lift"].get())
            self.sim.pc_stpt = float(self.pc_box.entries["P stpt (kPa)"].get())
            self.sim.pc_Kc = float(self.pc_box.entries["Kc"].get())
            self.sim.pc_tauI = float(self.pc_box.entries["taul (s)"].get())
        except ValueError:
            messagebox.showerror("Error", "Invalid input in Pressure Controller")

    def update_tc(self):
        try:
            self.sim.tc_mode = self.tc_box.mode_var.get()
            self.sim.Q = float(self.tc_box.entries["Q (kW)"].get())
            self.sim.tc_stpt = float(self.tc_box.entries["T stpt (K)"].get())
            self.sim.tc_Kc = float(self.tc_box.entries["Kc"].get())
            self.sim.tc_tauI = float(self.tc_box.entries["taul (s)"].get())
        except ValueError:
            messagebox.showerror("Error", "Invalid input in Temperature Controller")

    def update_lc(self):
        try:
            self.sim.lc_mode = self.lc_box.mode_var.get()
            self.sim.v_out = float(self.lc_box.entries["v_out (L/s)"].get())
            self.sim.lc_stpt = float(self.lc_box.entries["Lvl stpt (%)"].get())
            self.sim.lc_Kc = float(self.lc_box.entries["Kc"].get())
            self.sim.lc_tauI = float(self.lc_box.entries["taul (s)"].get())
        except ValueError:
            messagebox.showerror("Error", "Invalid input in Level Controller")

    def change_speed(self, val):
        self.sim_speed = float(val)
        self.speed_lbl.config(text=f"simulation speed: {int(self.sim_speed)}")

    def show_directions(self):
        messagebox.showinfo("Directions", "Flash Drum Control System Simulation\n\n"
                                          "1. Set inlet conditions on top-left.\n"
                                          "2. Adjust control parameters (Auto/Manual, Setpoint, Kc, TauI) for TC, PC, and LC.\n"
                                          "3. Click 'Update' on the respective frame to apply changes.\n"
                                          "4. Export live data using 'Snapshot .CSV'.")

    def save_csv(self):
        file_path = filedialog.asksaveasfilename(defaultextension=".csv", filetypes=[("CSV Files", "*.csv")])
        if file_path:
            with open(file_path, "w", newline="") as f:
                writer = csv.writer(f)
                writer.writerow(["Time (s)", "Temperature (K)", "Heat Duty (kW)", "Level (%)", "Bottoms Flow (L/s)", "Pressure (kPa)", "Valve Lift"])
                for t, T, Q, Lvl, v, P, lift in zip(self.sim.time_history, self.sim.T_history, self.sim.Q_history,
                                                    self.sim.Lvl_history, self.sim.v_out_history, self.sim.P_history, self.sim.lift_history):
                    writer.writerow([t, T, Q, Lvl, v, P, lift])
            messagebox.showinfo("Success", "Data saved successfully.")

    def update_loop(self):
        current_clock = time.perf_counter()
        elapsed_real_seconds = current_clock - self.last_update_time
        self.last_update_time = current_clock

        # Calculate time step (dt) scaled relative to real clock time and speed multiplier
        dt = elapsed_real_seconds * self.sim_speed

        # Advance physics engine using exact elapsed dt
        self.sim.step(dt=dt)

        # Update PFD drawing
        self.draw_pfd()

        # Update Plot Lines Efficiently
        if self.sim.time_history:
            times = [t - self.sim.current_time for t in self.sim.time_history]

            # Subplot 1
            self.line_T.set_data(times, self.sim.T_history)
            self.line_Q.set_data(times, self.sim.Q_history)
            self.ax1.relim(); self.ax1.autoscale_view()
            self.ax1_twin.relim(); self.ax1_twin.autoscale_view()

            # Subplot 2
            self.line_Lvl.set_data(times, self.sim.Lvl_history)
            self.line_vout.set_data(times, self.sim.v_out_history)
            self.ax2.relim(); self.ax2.autoscale_view()
            self.ax2_twin.relim(); self.ax2_twin.autoscale_view()

            # Subplot 3
            self.line_P.set_data(times, self.sim.P_history)
            self.line_lift.set_data(times, self.sim.lift_history)
            self.ax3.relim(); self.ax3.autoscale_view()
            self.ax3_twin.relim(); self.ax3_twin.autoscale_view()

            self.canvas_plot.draw_idle()

        # Re-trigger UI frame loop after ~50ms (~20 FPS update rate)
        self.root.after(50, self.update_loop)

if __name__ == "__main__":
    root = tk.Tk()
    app = FlashDrumGUI(root)
    root.mainloop()

## Prompt
You are an expert chemical engineer with years building programs. Can you code the front and back end for this chemical engineering software that solves a flasher system in python tkinter.
## Response
Here is a complete Python solution using tkinter and matplotlib that replicates both the UI layout and the underlying dynamic simulation/control backend depicted in the diagram.   
PNG

Python Implementation (flash_drum_sim.py)
Python
import time
import csv
import math
import tkinter as tk
from tkinter import ttk, messagebox, filedialog
import matplotlib
matplotlib.use("TkAgg")
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from matplotlib.figure import Figure

class FlashDrumSimulation:
    """Backend dynamic flash drum model and PID controllers."""
    def __init__(self):
        # Inlet variables
        self.F_in = 15.0       # mol/s
        self.x_butanol = 0.5   # mole fraction
        self.T_in = 400.0      # K

        # State variables
        self.T = 453.0         # Temperature (K)
        self.P = 69.8          # Pressure (kPa)
        self.Lvl = 18.7        # Liquid Level (%)
        self.Q = 500.0         # Heat duty (kW)
        self.lift = 0.5        # Vapor valve lift (0-1)
        self.v_out = 0.66      # Bottoms liquid flow rate (L/s)

        # Controllers Setup (Auto / Manual modes)
        # Pressure Controller (PC) -> manipulates Lift
        self.pc_mode = "Auto"
        self.pc_stpt = 74.6
        self.pc_Kc = 0.01
        self.pc_tauI = 36000.0
        self.pc_err_integral = 0.0

        # Temperature Controller (TC) -> manipulates Q
        self.tc_mode = "Auto"
        self.tc_stpt = 455.0
        self.tc_Kc = 10.0
        self.tc_tauI = 36000.0
        self.tc_err_integral = 0.0

        # Level Controller (LC) -> manipulates v_out
        self.lc_mode = "Auto"
        self.lc_stpt = 17.6
        self.lc_Kc = 0.05
        self.lc_tauI = 36000.0
        self.lc_err_integral = 0.0

        # Data History for Plots
        self.time_history = []
        self.T_history = []
        self.Q_history = []
        self.Lvl_history = []
        self.v_out_history = []
        self.P_history = []
        self.lift_history = []

        self.current_time = 0.0

    def step(self, dt=1.0):
        # 1. Update PID Controllers
        # PC
        if self.pc_mode == "Auto":
            err_P = self.P - self.pc_stpt
            self.pc_err_integral += err_P * dt
            i_term = (self.pc_Kc / self.pc_tauI) * self.pc_err_integral if self.pc_tauI > 0 else 0
            self.lift = max(0.0, min(1.0, 0.5 + self.pc_Kc * err_P + i_term))

        # TC
        if self.tc_mode == "Auto":
            err_T = self.tc_stpt - self.T
            self.tc_err_integral += err_T * dt
            i_term = (self.tc_Kc / self.tc_tauI) * self.tc_err_integral if self.tc_tauI > 0 else 0
            self.Q = max(0.0, min(2000.0, 500.0 + self.tc_Kc * err_T + i_term))

        # LC
        if self.lc_mode == "Auto":
            err_L = self.Lvl - self.lc_stpt
            self.lc_err_integral += err_L * dt
            i_term = (self.lc_Kc / self.lc_tauI) * self.lc_err_integral if self.lc_tauI > 0 else 0
            self.v_out = max(0.0, min(10.0, 0.66 + self.lc_Kc * err_L + i_term))

        # 2. Physics & Dynamic Balance Approximations
        # Simple dynamic responses modeling thermal, vapor, and liquid accumulation
        dT_dt = (self.Q - (self.T - self.T_in) * 2.0 - (self.T - 400.0) * 0.5) / 50.0
        self.T += dT_dt * dt

        # Pressure balance based on boiling and vapor valve outflow
        vapor_gen = max(0.0, (self.T - 420.0) * 1.5)
        vapor_out = self.lift * (self.P / 10.0) * 2.0
        dP_dt = (vapor_gen - vapor_out) / 2.0
        self.P = max(10.0, self.P + dP_dt * dt)

        # Level balance based on liquid accumulation vs bottoms discharge
        dLvl_dt = (self.F_in * (1.0 - self.x_butanol) * 0.1) - (self.v_out * 0.2)
        self.Lvl = max(0.0, min(100.0, self.Lvl + dLvl_dt * dt))

        # 3. Record History
        self.current_time += dt
        self.time_history.append(self.current_time)
        self.T_history.append(self.T)
        self.Q_history.append(self.Q)
        self.Lvl_history.append(self.Lvl)
        self.v_out_history.append(self.v_out)
        self.P_history.append(self.P)
        self.lift_history.append(self.lift)

        # Limit memory window (last 1000 seconds)
        if len(self.time_history) > 1000:
            self.time_history.pop(0)
            self.T_history.pop(0)
            self.Q_history.pop(0)
            self.Lvl_history.pop(0)
            self.v_out_history.pop(0)
            self.P_history.pop(0)
            self.lift_history.pop(0)


class FlashDrumGUI:
    def __init__(self, root):
        self.root = root
        self.root.title("Aniline-Butanol Flash Drum Simulator")
        self.root.geometry("1100x850")
        self.root.configure(bg="#e8e8e8")

        self.sim = FlashDrumSimulation()
        self.sim_speed = 2

        self.setup_ui()
        self.update_loop()

    def setup_ui(self):
        # Top Header Frame
        top_frame = tk.Frame(self.root, bg="#e8e8e8")
        top_frame.pack(side=tk.TOP, fill=tk.X, padx=10, pady=5)

        # Inlet Controls Frame
        inlet_box = tk.LabelFrame(top_frame, text="inlet conditions", font=("Arial", 11, "bold"), bg="white")
        inlet_box.pack(side=tk.LEFT, padx=10)

        tk.Button(inlet_box, text="Update", bg="#007bff", fg="white", font=("Arial", 9, "bold"),
                  command=self.update_inlets).grid(row=0, column=0, columnspan=2, pady=2, sticky="ew")

        tk.Label(inlet_box, text="F_in (mol/s)", bg="white").grid(row=1, column=0, sticky="w", padx=5)
        self.entry_Fin = tk.Entry(inlet_box, width=8)
        self.entry_Fin.insert(0, "15")
        self.entry_Fin.grid(row=1, column=1, padx=5, pady=2)

        tk.Label(inlet_box, text="x_butanol", bg="white").grid(row=2, column=0, sticky="w", padx=5)
        self.entry_xbut = tk.Entry(inlet_box, width=8)
        self.entry_xbut.insert(0, "0.5")
        self.entry_xbut.grid(row=2, column=1, padx=5, pady=2)

        tk.Label(inlet_box, text="T_in (K)", bg="white").grid(row=3, column=0, sticky="w", padx=5)
        self.entry_Tin = tk.Entry(inlet_box, width=8)
        self.entry_Tin.insert(0, "400")
        self.entry_Tin.grid(row=3, column=1, padx=5, pady=2)

        # Speed, Snapshot, Directions
        ctrl_box = tk.Frame(top_frame, bg="#e8e8e8")
        ctrl_box.pack(side=tk.LEFT, padx=20)

        speed_frame = tk.Frame(ctrl_box, bg="#d0e0e3", bd=1, relief="solid")
        speed_frame.pack(pady=2, fill=tk.X)
        self.speed_lbl = tk.Label(speed_frame, text=f"simulation speed: {self.sim_speed}", bg="#d0e0e3", font=("Arial", 10))
        self.speed_lbl.pack()
        self.speed_slider = ttk.Scale(speed_frame, from_=1, to=10, value=2, command=self.change_speed)
        self.speed_slider.pack(padx=5, pady=2)

        tk.Button(ctrl_box, text="Snapshot .CSV", bg="#28a745", fg="white", font=("Arial", 10, "bold"),
                  command=self.save_csv).pack(pady=4, fill=tk.X)
        tk.Button(ctrl_box, text="Directions", bg="#5a6268", fg="white", font=("Arial", 10, "bold"),
                  command=self.show_directions).pack(pady=2, fill=tk.X)

        # Diagram & Controller Box Area
        center_frame = tk.Frame(self.root, bg="#e8e8e8")
        center_frame.pack(side=tk.TOP, fill=tk.BOTH, expand=True, padx=10, pady=5)

        # Left Controllers (TC)
        left_ctrl_frame = tk.Frame(center_frame, bg="#e8e8e8")
        left_ctrl_frame.pack(side=tk.LEFT, anchor="n", padx=10)

        self.tc_box = self.create_controller_ui(left_ctrl_frame, "temperature controller", "Q (kW)", "T stpt (K)",
                                                 "450", "455", "0", "36000", self.update_tc)

        # Center PFD Schematic (Canvas)
        self.canvas = tk.Canvas(center_frame, width=320, height=300, bg="#e8e8e8", highlightthickness=0)
        self.canvas.pack(side=tk.LEFT, expand=True)
        self.draw_pfd()

        # Right Controllers (PC and LC)
        right_ctrl_frame = tk.Frame(center_frame, bg="#e8e8e8")
        right_ctrl_frame.pack(side=tk.RIGHT, anchor="n", padx=10)

        self.pc_box = self.create_controller_ui(right_ctrl_frame, "pressure controller", "Lift", "P stpt (kPa)",
                                                 "0.5", "74.6", "0", "36000", self.update_pc)
        self.lc_box = self.create_controller_ui(right_ctrl_frame, "level controller", "v_out (L/s)", "Lvl stpt (%)",
                                                 "0.66", "17.6", "0", "36000", self.update_lc)

        # Bottom Plots Frame
        plot_frame = tk.Frame(self.root, bg="white")
        plot_frame.pack(side=tk.BOTTOM, fill=tk.BOTH, expand=True, padx=10, pady=10)

        self.fig = Figure(figsize=(10, 2.8), dpi=100)
        self.ax1 = self.fig.add_subplot(131)
        self.ax2 = self.fig.add_subplot(132)
        self.ax3 = self.fig.add_subplot(133)
        self.fig.tight_layout(pad=2.0)

        self.canvas_plot = FigureCanvasTkAgg(self.fig, master=plot_frame)
        self.canvas_plot.get_tk_widget().pack(fill=tk.BOTH, expand=True)

    def create_controller_ui(self, parent, title, mv_name, stpt_name, mv_val, stpt_val, kc_val, tau_val, update_cmd):
        box = tk.LabelFrame(parent, text=title, font=("Arial", 11, "bold"), bg="white")
        box.pack(pady=5, fill=tk.X)

        mode_frame = tk.Frame(box, bg="white")
        mode_frame.grid(row=0, column=0, columnspan=2, pady=2, sticky="ew")

        mode_var = tk.StringVar(value="Auto")
        auto_btn = tk.Radiobutton(mode_frame, text="Auto", value="Auto", variable=mode_var, indicatoron=False, width=6, bg="#6c757d", selectcolor="#007bff", fg="white")
        man_btn = tk.Radiobutton(mode_frame, text="Manual", value="Manual", variable=mode_var, indicatoron=False, width=6, bg="#6c757d", selectcolor="#007bff", fg="white")
        auto_btn.pack(side=tk.LEFT, padx=2)
        man_btn.pack(side=tk.LEFT, padx=2)

        tk.Button(mode_frame, text="Update", bg="#007bff", fg="white", font=("Arial", 8, "bold"), command=update_cmd).pack(side=tk.RIGHT, padx=2)

        labels = [mv_name, stpt_name, "Kc", "taul (s)"]
        defaults = [mv_val, stpt_val, kc_val, tau_val]
        entries = {}

        for idx, (lbl, df) in enumerate(zip(labels, defaults), start=1):
            tk.Label(box, text=lbl, bg="white", fg="blue" if idx==1 else ("red" if idx==2 else "black")).grid(row=idx, column=0, sticky="w", padx=5)
            e = tk.Entry(box, width=8)
            e.insert(0, df)
            e.grid(row=idx, column=1, padx=5, pady=2)
            entries[lbl] = e

        box.mode_var = mode_var
        box.entries = entries
        return box

    def draw_pfd(self):
        self.canvas.delete("all")
        # Flash vessel body
        self.canvas.create_rectangle(120, 50, 200, 220, fill="#d3d3d3", outline="black", width=2)
        self.canvas.create_arc(120, 30, 200, 70, start=0, extent=180, fill="#d3d3d3", outline="black", width=2)
        self.canvas.create_arc(120, 200, 200, 240, start=180, extent=180, fill="#d3d3d3", outline="black", width=2)

        # Liquid level in drum
        lvl_h = max(5, int((self.sim.Lvl / 100.0) * 120))
        self.canvas.create_rectangle(121, 220 - lvl_h, 199, 220, fill="#d2b48c", outline="")

        # Text on drum
        self.canvas.create_text(160, 90, text="aniline\nbutanol", font=("Arial", 10, "bold"))

        # Controller badges & Dynamic values
        # PC
        self.canvas.create_oval(230, 30, 270, 70, fill="white", outline="black", width=2)
        self.canvas.create_text(250, 50, text="PC", font=("Arial", 11, "bold"))
        self.canvas.create_text(250, 20, text=f"{self.sim.P:.1f} kPa", fill="green", font=("Arial", 10, "bold"))

        # LC
        self.canvas.create_oval(230, 150, 270, 190, fill="white", outline="black", width=2)
        self.canvas.create_text(250, 170, text="LC", font=("Arial", 11, "bold"))
        self.canvas.create_text(250, 130, text=f"{self.sim.Lvl:.1f} %", fill="green", font=("Arial", 10, "bold"))

        # TC
        self.canvas.create_oval(50, 150, 90, 190, fill="white", outline="black", width=2)
        self.canvas.create_text(70, 170, text="TC", font=("Arial", 11, "bold"))
        self.canvas.create_text(70, 210, text=f"{self.sim.T:.0f} K", fill="green", font=("Arial", 10, "bold"))

    def update_inlets(self):
        try:
            self.sim.F_in = float(self.entry_Fin.get())
            self.sim.x_butanol = float(self.entry_xbut.get())
            self.sim.T_in = float(self.entry_Tin.get())
        except ValueError:
            messagebox.showerror("Error", "Invalid numeric inlet conditions")

    def update_pc(self):
        try:
            self.sim.pc_mode = self.pc_box.mode_var.get()
            self.sim.lift = float(self.pc_box.entries["Lift"].get())
            self.sim.pc_stpt = float(self.pc_box.entries["P stpt (kPa)"].get())
            self.sim.pc_Kc = float(self.pc_box.entries["Kc"].get())
            self.sim.pc_tauI = float(self.pc_box.entries["taul (s)"].get())
        except ValueError:
            messagebox.showerror("Error", "Invalid input in Pressure Controller")

    def update_tc(self):
        try:
            self.sim.tc_mode = self.tc_box.mode_var.get()
            self.sim.Q = float(self.tc_box.entries["Q (kW)"].get())
            self.sim.tc_stpt = float(self.tc_box.entries["T stpt (K)"].get())
            self.sim.tc_Kc = float(self.tc_box.entries["Kc"].get())
            self.sim.tc_tauI = float(self.tc_box.entries["taul (s)"].get())
        except ValueError:
            messagebox.showerror("Error", "Invalid input in Temperature Controller")

    def update_lc(self):
        try:
            self.sim.lc_mode = self.lc_box.mode_var.get()
            self.sim.v_out = float(self.lc_box.entries["v_out (L/s)"].get())
            self.sim.lc_stpt = float(self.lc_box.entries["Lvl stpt (%)"].get())
            self.sim.lc_Kc = float(self.lc_box.entries["Kc"].get())
            self.sim.lc_tauI = float(self.lc_box.entries["taul (s)"].get())
        except ValueError:
            messagebox.showerror("Error", "Invalid input in Level Controller")

    def change_speed(self, val):
        self.sim_speed = int(float(val))
        self.speed_lbl.config(text=f"simulation speed: {self.sim_speed}")

    def show_directions(self):
        messagebox.showinfo("Directions", "Flash Drum Control System Simulation\n\n"
                                          "1. Set inlet conditions on top-left.\n"
                                          "2. Adjust control parameters (Auto/Manual, Setpoint, Kc, TauI) for TC, PC, and LC.\n"
                                          "3. Click 'Update' on the respective frame to apply changes.\n"
                                          "4. Export live data using 'Snapshot .CSV'.")

    def save_csv(self):
        file_path = filedialog.asksaveasfilename(defaultextension=".csv", filetypes=[("CSV Files", "*.csv")])
        if file_path:
            with open(file_path, "w", newline="") as f:
                writer = csv.writer(f)
                writer.writerow(["Time (s)", "Temperature (K)", "Heat Duty (kW)", "Level (%)", "Bottoms Flow (L/s)", "Pressure (kPa)", "Valve Lift"])
                for t, T, Q, Lvl, v, P, lift in zip(self.sim.time_history, self.sim.T_history, self.sim.Q_history,
                                                    self.sim.Lvl_history, self.sim.v_out_history, self.sim.P_history, self.sim.lift_history):
                    writer.writerow([t, T, Q, Lvl, v, P, lift])
            messagebox.showinfo("Success", "Data saved successfully.")

    def update_loop(self):
        # Perform simulation steps based on speed
        for _ in range(self.sim_speed):
            self.sim.step(dt=1.0)

        # Redraw PFD Dynamic Indicators
        self.draw_pfd()

        # Refresh Plots
        times = [t - self.sim.current_time for t in self.sim.time_history] # relative time window

        self.ax1.clear()
        self.ax1.plot(times, self.sim.T_history, 'g-', label="Temp (K)")
        ax1_twin = self.ax1.twinx()
        ax1_twin.plot(times, self.sim.Q_history, 'b-', label="Q (kW)")
        self.ax1.set_title("temperature and heat duty", fontsize=9)
        self.ax1.set_xlabel("time (s)", fontsize=8)

        self.ax2.clear()
        self.ax2.plot(times, self.sim.Lvl_history, 'g-', label="Level (%)")
        ax2_twin = self.ax2.twinx()
        ax2_twin.plot(times, self.sim.v_out_history, 'b-', label="Flow (L/s)")
        self.ax2.set_title("level and bottoms flow rate", fontsize=9)
        self.ax2.set_xlabel("time (s)", fontsize=8)

        self.ax3.clear()
        self.ax3.plot(times, self.sim.P_history, 'g-', label="Pressure (kPa)")
        ax3_twin = self.ax3.twinx()
        ax3_twin.plot(times, self.sim.lift_history, 'b-', label="Lift")
        self.ax3.set_title("pressure and valve lift", fontsize=9)
        self.ax3.set_xlabel("time (s)", fontsize=8)

        self.canvas_plot.draw()

        # Loop again after 500ms
        self.root.after(500, self.update_loop)

if __name__ == "__main__":
    root = tk.Tk()
    app = FlashDrumGUI(root)
    root.mainloop()
Dependencies
Run pip install matplotlib before executing the script.
## Prompt
The matplotlib plots do not update properly in this code can you correct it and write the corrections in a new py file
## Response
Here is the corrected Python script. The primary issue in the original code was calling ax.clear() and creating new twinx() axes inside the loop on every refresh cycle. Creating duplicate overlay axes repeatedly causes performance memory leaks and breaks line updates.

This revised version creates fixed twinx() axes during setup and updates the line data directly using set_data() and relim()/autoscale_view(), ensuring smooth real-time plotting.

Save this code in a file named flash_drum_sim_fixed.py:

Python
import time
import csv
import math
import tkinter as tk
from tkinter import ttk, messagebox, filedialog
import matplotlib
matplotlib.use("TkAgg")
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from matplotlib.figure import Figure

class FlashDrumSimulation:
    """Backend dynamic flash drum model and PID controllers."""
    def __init__(self):
        # Inlet variables
        self.F_in = 15.0       # mol/s
        self.x_butanol = 0.5   # mole fraction
        self.T_in = 400.0      # K

        # State variables
        self.T = 453.0         # Temperature (K)
        self.P = 69.8          # Pressure (kPa)
        self.Lvl = 18.7        # Liquid Level (%)
        self.Q = 500.0         # Heat duty (kW)
        self.lift = 0.5        # Vapor valve lift (0-1)
        self.v_out = 0.66      # Bottoms liquid flow rate (L/s)

        # Controllers Setup (Auto / Manual modes)
        # Pressure Controller (PC) -> manipulates Lift
        self.pc_mode = "Auto"
        self.pc_stpt = 74.6
        self.pc_Kc = 0.01
        self.pc_tauI = 36000.0
        self.pc_err_integral = 0.0

        # Temperature Controller (TC) -> manipulates Q
        self.tc_mode = "Auto"
        self.tc_stpt = 455.0
        self.tc_Kc = 10.0
        self.tc_tauI = 36000.0
        self.tc_err_integral = 0.0

        # Level Controller (LC) -> manipulates v_out
        self.lc_mode = "Auto"
        self.lc_stpt = 17.6
        self.lc_Kc = 0.05
        self.lc_tauI = 36000.0
        self.lc_err_integral = 0.0

        # Data History for Plots
        self.time_history = []
        self.T_history = []
        self.Q_history = []
        self.Lvl_history = []
        self.v_out_history = []
        self.P_history = []
        self.lift_history = []

        self.current_time = 0.0

    def step(self, dt=1.0):
        # 1. Update PID Controllers
        # PC
        if self.pc_mode == "Auto":
            err_P = self.P - self.pc_stpt
            self.pc_err_integral += err_P * dt
            i_term = (self.pc_Kc / self.pc_tauI) * self.pc_err_integral if self.pc_tauI > 0 else 0
            self.lift = max(0.0, min(1.0, 0.5 + self.pc_Kc * err_P + i_term))

        # TC
        if self.tc_mode == "Auto":
            err_T = self.tc_stpt - self.T
            self.tc_err_integral += err_T * dt
            i_term = (self.tc_Kc / self.tc_tauI) * self.tc_err_integral if self.tc_tauI > 0 else 0
            self.Q = max(0.0, min(2000.0, 500.0 + self.tc_Kc * err_T + i_term))

        # LC
        if self.lc_mode == "Auto":
            err_L = self.Lvl - self.lc_stpt
            self.lc_err_integral += err_L * dt
            i_term = (self.lc_Kc / self.lc_tauI) * self.lc_err_integral if self.lc_tauI > 0 else 0
            self.v_out = max(0.0, min(10.0, 0.66 + self.lc_Kc * err_L + i_term))

        # 2. Physics & Dynamic Balance Approximations
        dT_dt = (self.Q - (self.T - self.T_in) * 2.0 - (self.T - 400.0) * 0.5) / 50.0
        self.T += dT_dt * dt

        vapor_gen = max(0.0, (self.T - 420.0) * 1.5)
        vapor_out = self.lift * (self.P / 10.0) * 2.0
        dP_dt = (vapor_gen - vapor_out) / 2.0
        self.P = max(10.0, self.P + dP_dt * dt)

        dLvl_dt = (self.F_in * (1.0 - self.x_butanol) * 0.1) - (self.v_out * 0.2)
        self.Lvl = max(0.0, min(100.0, self.Lvl + dLvl_dt * dt))

        # 3. Record History
        self.current_time += dt
        self.time_history.append(self.current_time)
        self.T_history.append(self.T)
        self.Q_history.append(self.Q)
        self.Lvl_history.append(self.Lvl)
        self.v_out_history.append(self.v_out)
        self.P_history.append(self.P)
        self.lift_history.append(self.lift)

        # Limit memory window (last 300 points)
        if len(self.time_history) > 300:
            self.time_history.pop(0)
            self.T_history.pop(0)
            self.Q_history.pop(0)
            self.Lvl_history.pop(0)
            self.v_out_history.pop(0)
            self.P_history.pop(0)
            self.lift_history.pop(0)


class FlashDrumGUI:
    def __init__(self, root):
        self.root = root
        self.root.title("Aniline-Butanol Flash Drum Simulator")
        self.root.geometry("1100x850")
        self.root.configure(bg="#e8e8e8")

        self.sim = FlashDrumSimulation()
        self.sim_speed = 2

        self.setup_ui()
        self.setup_plots()
        self.update_loop()

    def setup_ui(self):
        # Top Header Frame
        top_frame = tk.Frame(self.root, bg="#e8e8e8")
        top_frame.pack(side=tk.TOP, fill=tk.X, padx=10, pady=5)

        # Inlet Controls Frame
        inlet_box = tk.LabelFrame(top_frame, text="inlet conditions", font=("Arial", 11, "bold"), bg="white")
        inlet_box.pack(side=tk.LEFT, padx=10)

        tk.Button(inlet_box, text="Update", bg="#007bff", fg="white", font=("Arial", 9, "bold"),
                  command=self.update_inlets).grid(row=0, column=0, columnspan=2, pady=2, sticky="ew")

        tk.Label(inlet_box, text="F_in (mol/s)", bg="white").grid(row=1, column=0, sticky="w", padx=5)
        self.entry_Fin = tk.Entry(inlet_box, width=8)
        self.entry_Fin.insert(0, "15")
        self.entry_Fin.grid(row=1, column=1, padx=5, pady=2)

        tk.Label(inlet_box, text="x_butanol", bg="white").grid(row=2, column=0, sticky="w", padx=5)
        self.entry_xbut = tk.Entry(inlet_box, width=8)
        self.entry_xbut.insert(0, "0.5")
        self.entry_xbut.grid(row=2, column=1, padx=5, pady=2)

        tk.Label(inlet_box, text="T_in (K)", bg="white").grid(row=3, column=0, sticky="w", padx=5)
        self.entry_Tin = tk.Entry(inlet_box, width=8)
        self.entry_Tin.insert(0, "400")
        self.entry_Tin.grid(row=3, column=1, padx=5, pady=2)

        # Speed, Snapshot, Directions
        ctrl_box = tk.Frame(top_frame, bg="#e8e8e8")
        ctrl_box.pack(side=tk.LEFT, padx=20)

        speed_frame = tk.Frame(ctrl_box, bg="#d0e0e3", bd=1, relief="solid")
        speed_frame.pack(pady=2, fill=tk.X)
        self.speed_lbl = tk.Label(speed_frame, text=f"simulation speed: {self.sim_speed}", bg="#d0e0e3", font=("Arial", 10))
        self.speed_lbl.pack()
        self.speed_slider = ttk.Scale(speed_frame, from_=1, to=10, value=2, command=self.change_speed)
        self.speed_slider.pack(padx=5, pady=2)

        tk.Button(ctrl_box, text="Snapshot .CSV", bg="#28a745", fg="white", font=("Arial", 10, "bold"),
                  command=self.save_csv).pack(pady=4, fill=tk.X)
        tk.Button(ctrl_box, text="Directions", bg="#5a6268", fg="white", font=("Arial", 10, "bold"),
                  command=self.show_directions).pack(pady=2, fill=tk.X)

        # Diagram & Controller Box Area
        center_frame = tk.Frame(self.root, bg="#e8e8e8")
        center_frame.pack(side=tk.TOP, fill=tk.BOTH, expand=True, padx=10, pady=5)

        # Left Controllers (TC)
        left_ctrl_frame = tk.Frame(center_frame, bg="#e8e8e8")
        left_ctrl_frame.pack(side=tk.LEFT, anchor="n", padx=10)

        self.tc_box = self.create_controller_ui(left_ctrl_frame, "temperature controller", "Q (kW)", "T stpt (K)",
                                                 "500", "455", "0", "36000", self.update_tc)

        # Center PFD Schematic (Canvas)
        self.canvas = tk.Canvas(center_frame, width=320, height=260, bg="#e8e8e8", highlightthickness=0)
        self.canvas.pack(side=tk.LEFT, expand=True)
        self.draw_pfd()

        # Right Controllers (PC and LC)
        right_ctrl_frame = tk.Frame(center_frame, bg="#e8e8e8")
        right_ctrl_frame.pack(side=tk.RIGHT, anchor="n", padx=10)

        self.pc_box = self.create_controller_ui(right_ctrl_frame, "pressure controller", "Lift", "P stpt (kPa)",
                                                 "0.5", "74.6", "0", "36000", self.update_pc)
        self.lc_box = self.create_controller_ui(right_ctrl_frame, "level controller", "v_out (L/s)", "Lvl stpt (%)",
                                                 "0.66", "17.6", "0", "36000", self.update_lc)

    def create_controller_ui(self, parent, title, mv_name, stpt_name, mv_val, stpt_val, kc_val, tau_val, update_cmd):
        box = tk.LabelFrame(parent, text=title, font=("Arial", 11, "bold"), bg="white")
        box.pack(pady=5, fill=tk.X)

        mode_frame = tk.Frame(box, bg="white")
        mode_frame.grid(row=0, column=0, columnspan=2, pady=2, sticky="ew")

        mode_var = tk.StringVar(value="Auto")
        auto_btn = tk.Radiobutton(mode_frame, text="Auto", value="Auto", variable=mode_var, indicatoron=False, width=6, bg="#6c757d", selectcolor="#007bff", fg="white")
        man_btn = tk.Radiobutton(mode_frame, text="Manual", value="Manual", variable=mode_var, indicatoron=False, width=6, bg="#6c757d", selectcolor="#007bff", fg="white")
        auto_btn.pack(side=tk.LEFT, padx=2)
        man_btn.pack(side=tk.LEFT, padx=2)

        tk.Button(mode_frame, text="Update", bg="#007bff", fg="white", font=("Arial", 8, "bold"), command=update_cmd).pack(side=tk.RIGHT, padx=2)

        labels = [mv_name, stpt_name, "Kc", "taul (s)"]
        defaults = [mv_val, stpt_val, kc_val, tau_val]
        entries = {}

        for idx, (lbl, df) in enumerate(zip(labels, defaults), start=1):
            tk.Label(box, text=lbl, bg="white", fg="blue" if idx==1 else ("red" if idx==2 else "black")).grid(row=idx, column=0, sticky="w", padx=5)
            e = tk.Entry(box, width=8)
            e.insert(0, df)
            e.grid(row=idx, column=1, padx=5, pady=2)
            entries[lbl] = e

        box.mode_var = mode_var
        box.entries = entries
        return box

    def setup_plots(self):
        plot_frame = tk.Frame(self.root, bg="white")
        plot_frame.pack(side=tk.BOTTOM, fill=tk.BOTH, expand=True, padx=10, pady=10)

        self.fig = Figure(figsize=(10, 3.0), dpi=100)
        
        # Subplot 1: Temp & Heat Duty
        self.ax1 = self.fig.add_subplot(131)
        self.ax1_twin = self.ax1.twinx()
        self.line_T, = self.ax1.plot([], [], 'g-', label="Temp (K)")
        self.line_Q, = self.ax1_twin.plot([], [], 'b-', label="Q (kW)")
        self.ax1.set_title("temperature and heat duty", fontsize=9)
        self.ax1.set_xlabel("time (s)", fontsize=8)
        self.ax1.set_ylabel("temperature (K)", color='g', fontsize=8)
        self.ax1_twin.set_ylabel("heat duty (kW)", color='b', fontsize=8)

        # Subplot 2: Level & Flow
        self.ax2 = self.fig.add_subplot(132)
        self.ax2_twin = self.ax2.twinx()
        self.line_Lvl, = self.ax2.plot([], [], 'g-', label="Level (%)")
        self.line_vout, = self.ax2_twin.plot([], [], 'b-', label="Flow (L/s)")
        self.ax2.set_title("level and bottoms flow rate", fontsize=9)
        self.ax2.set_xlabel("time (s)", fontsize=8)
        self.ax2.set_ylabel("level (%)", color='g', fontsize=8)
        self.ax2_twin.set_ylabel("bottoms flow (L / s)", color='b', fontsize=8)

        # Subplot 3: Pressure & Valve Lift
        self.ax3 = self.fig.add_subplot(133)
        self.ax3_twin = self.ax3.twinx()
        self.line_P, = self.ax3.plot([], [], 'g-', label="Pressure (kPa)")
        self.line_lift, = self.ax3_twin.plot([], [], 'b-', label="Lift")
        self.ax3.set_title("pressure and valve lift", fontsize=9)
        self.ax3.set_xlabel("time (s)", fontsize=8)
        self.ax3.set_ylabel("pressure (kPa)", color='g', fontsize=8)
        self.ax3_twin.set_ylabel("valve lift", color='b', fontsize=8)

        self.fig.tight_layout(pad=2.0)
        self.canvas_plot = FigureCanvasTkAgg(self.fig, master=plot_frame)
        self.canvas_plot.get_tk_widget().pack(fill=tk.BOTH, expand=True)

    def draw_pfd(self):
        self.canvas.delete("all")
        # Drum body
        self.canvas.create_rectangle(120, 50, 200, 200, fill="#d3d3d3", outline="black", width=2)
        self.canvas.create_arc(120, 30, 200, 70, start=0, extent=180, fill="#d3d3d3", outline="black", width=2)
        self.canvas.create_arc(120, 180, 200, 220, start=180, extent=180, fill="#d3d3d3", outline="black", width=2)

        # Liquid level in drum
        lvl_h = max(5, int((self.sim.Lvl / 100.0) * 120))
        self.canvas.create_rectangle(121, 200 - lvl_h, 199, 200, fill="#d2b48c", outline="")

        # Label on drum
        self.canvas.create_text(160, 90, text="aniline\nbutanol", font=("Arial", 10, "bold"))

        # Controller badges & dynamic numbers
        self.canvas.create_oval(230, 30, 270, 70, fill="white", outline="black", width=2)
        self.canvas.create_text(250, 50, text="PC", font=("Arial", 11, "bold"))
        self.canvas.create_text(250, 18, text=f"{self.sim.P:.1f} kPa", fill="green", font=("Arial", 10, "bold"))

        self.canvas.create_oval(230, 140, 270, 180, fill="white", outline="black", width=2)
        self.canvas.create_text(250, 160, text="LC", font=("Arial", 11, "bold"))
        self.canvas.create_text(250, 122, text=f"{self.sim.Lvl:.1f} %", fill="green", font=("Arial", 10, "bold"))

        self.canvas.create_oval(50, 140, 90, 180, fill="white", outline="black", width=2)
        self.canvas.create_text(70, 160, text="TC", font=("Arial", 11, "bold"))
        self.canvas.create_text(70, 198, text=f"{self.sim.T:.0f} K", fill="green", font=("Arial", 10, "bold"))

    def update_inlets(self):
        try:
            self.sim.F_in = float(self.entry_Fin.get())
            self.sim.x_butanol = float(self.entry_xbut.get())
            self.sim.T_in = float(self.entry_Tin.get())
        except ValueError:
            messagebox.showerror("Error", "Invalid numeric inlet conditions")

    def update_pc(self):
        try:
            self.sim.pc_mode = self.pc_box.mode_var.get()
            self.sim.lift = float(self.pc_box.entries["Lift"].get())
            self.sim.pc_stpt = float(self.pc_box.entries["P stpt (kPa)"].get())
            self.sim.pc_Kc = float(self.pc_box.entries["Kc"].get())
            self.sim.pc_tauI = float(self.pc_box.entries["taul (s)"].get())
        except ValueError:
            messagebox.showerror("Error", "Invalid input in Pressure Controller")

    def update_tc(self):
        try:
            self.sim.tc_mode = self.tc_box.mode_var.get()
            self.sim.Q = float(self.tc_box.entries["Q (kW)"].get())
            self.sim.tc_stpt = float(self.tc_box.entries["T stpt (K)"].get())
            self.sim.tc_Kc = float(self.tc_box.entries["Kc"].get())
            self.sim.tc_tauI = float(self.tc_box.entries["taul (s)"].get())
        except ValueError:
            messagebox.showerror("Error", "Invalid input in Temperature Controller")

    def update_lc(self):
        try:
            self.sim.lc_mode = self.lc_box.mode_var.get()
            self.sim.v_out = float(self.lc_box.entries["v_out (L/s)"].get())
            self.sim.lc_stpt = float(self.lc_box.entries["Lvl stpt (%)"].get())
            self.sim.lc_Kc = float(self.lc_box.entries["Kc"].get())
            self.sim.lc_tauI = float(self.lc_box.entries["taul (s)"].get())
        except ValueError:
            messagebox.showerror("Error", "Invalid input in Level Controller")

    def change_speed(self, val):
        self.sim_speed = int(float(val))
        self.speed_lbl.config(text=f"simulation speed: {self.sim_speed}")

    def show_directions(self):
        messagebox.showinfo("Directions", "Flash Drum Control System Simulation\n\n"
                                          "1. Set inlet conditions on top-left.\n"
                                          "2. Adjust control parameters (Auto/Manual, Setpoint, Kc, TauI) for TC, PC, and LC.\n"
                                          "3. Click 'Update' on the respective frame to apply changes.\n"
                                          "4. Export live data using 'Snapshot .CSV'.")

    def save_csv(self):
        file_path = filedialog.asksaveasfilename(defaultextension=".csv", filetypes=[("CSV Files", "*.csv")])
        if file_path:
            with open(file_path, "w", newline="") as f:
                writer = csv.writer(f)
                writer.writerow(["Time (s)", "Temperature (K)", "Heat Duty (kW)", "Level (%)", "Bottoms Flow (L/s)", "Pressure (kPa)", "Valve Lift"])
                for t, T, Q, Lvl, v, P, lift in zip(self.sim.time_history, self.sim.T_history, self.sim.Q_history,
                                                    self.sim.Lvl_history, self.sim.v_out_history, self.sim.P_history, self.sim.lift_history):
                    writer.writerow([t, T, Q, Lvl, v, P, lift])
            messagebox.showinfo("Success", "Data saved successfully.")

    def update_loop(self):
        # Step simulation
        for _ in range(self.sim_speed):
            self.sim.step(dt=1.0)

        # Update PFD drawing
        self.draw_pfd()

        # Update Plot Lines Efficiently
        if self.sim.time_history:
            times = [t - self.sim.current_time for t in self.sim.time_history]

            # Subplot 1
            self.line_T.set_data(times, self.sim.T_history)
            self.line_Q.set_data(times, self.sim.Q_history)
            self.ax1.relim(); self.ax1.autoscale_view()
            self.ax1_twin.relim(); self.ax1_twin.autoscale_view()

            # Subplot 2
            self.line_Lvl.set_data(times, self.sim.Lvl_history)
            self.line_vout.set_data(times, self.sim.v_out_history)
            self.ax2.relim(); self.ax2.autoscale_view()
            self.ax2_twin.relim(); self.ax2_twin.autoscale_view()

            # Subplot 3
            self.line_P.set_data(times, self.sim.P_history)
            self.line_lift.set_data(times, self.sim.lift_history)
            self.ax3.relim(); self.ax3.autoscale_view()
            self.ax3_twin.relim(); self.ax3_twin.autoscale_view()

            self.canvas_plot.draw_idle()

        # Refresh rate
        self.root.after(300, self.update_loop)

if __name__ == "__main__":
    root = tk.Tk()
    app = FlashDrumGUI(root)
    root.mainloop()
Core Fixes Applied
Persistent Axis References: Created ax.twinx() secondary axes once in setup_plots() rather than calling twinx() on every update frame.

Direct Line Updating: Replaced ax.clear() with line.set_data(), which updates existing Matplotlib line elements dynamically.

Optimized Canvas Rendering: Changed canvas.draw() to canvas.draw_idle(), preventing UI lag and canvas freeze issues during simulation steps.
## Prompt
This code is good but the timer for simulation seems to be tied to cpu speed, can you make it retaliative to real speed. Rewrite the fix in a new py file
## Response
Here is the corrected code where the simulation step size (dt) and timer are decoupled from the CPU execution rate and calculated relative to actual elapsed real time using time.perf_counter().

Save this revised implementation in a new file named flash_drum_sim_realtime.py:

Python
import time
import csv
import math
import tkinter as tk
from tkinter import ttk, messagebox, filedialog
import matplotlib
matplotlib.use("TkAgg")
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from matplotlib.figure import Figure

class FlashDrumSimulation:
    """Backend dynamic flash drum model and PID controllers."""
    def __init__(self):
        # Inlet variables
        self.F_in = 15.0       # mol/s
        self.x_butanol = 0.5   # mole fraction
        self.T_in = 400.0      # K

        # State variables
        self.T = 453.0         # Temperature (K)
        self.P = 69.8          # Pressure (kPa)
        self.Lvl = 18.7        # Liquid Level (%)
        self.Q = 500.0         # Heat duty (kW)
        self.lift = 0.5        # Vapor valve lift (0-1)
        self.v_out = 0.66      # Bottoms liquid flow rate (L/s)

        # Controllers Setup (Auto / Manual modes)
        # Pressure Controller (PC) -> manipulates Lift
        self.pc_mode = "Auto"
        self.pc_stpt = 74.6
        self.pc_Kc = 0.01
        self.pc_tauI = 36000.0
        self.pc_err_integral = 0.0

        # Temperature Controller (TC) -> manipulates Q
        self.tc_mode = "Auto"
        self.tc_stpt = 455.0
        self.tc_Kc = 10.0
        self.tc_tauI = 36000.0
        self.tc_err_integral = 0.0

        # Level Controller (LC) -> manipulates v_out
        self.lc_mode = "Auto"
        self.lc_stpt = 17.6
        self.lc_Kc = 0.05
        self.lc_tauI = 36000.0
        self.lc_err_integral = 0.0

        # Data History for Plots
        self.time_history = []
        self.T_history = []
        self.Q_history = []
        self.Lvl_history = []
        self.v_out_history = []
        self.P_history = []
        self.lift_history = []

        self.current_time = 0.0

    def step(self, dt):
        """Advances the differential equations based on real elapsed time delta dt."""
        if dt <= 0:
            return

        # 1. Update PID Controllers
        # PC
        if self.pc_mode == "Auto":
            err_P = self.P - self.pc_stpt
            self.pc_err_integral += err_P * dt
            i_term = (self.pc_Kc / self.pc_tauI) * self.pc_err_integral if self.pc_tauI > 0 else 0
            self.lift = max(0.0, min(1.0, 0.5 + self.pc_Kc * err_P + i_term))

        # TC
        if self.tc_mode == "Auto":
            err_T = self.tc_stpt - self.T
            self.tc_err_integral += err_T * dt
            i_term = (self.tc_Kc / self.tc_tauI) * self.tc_err_integral if self.tc_tauI > 0 else 0
            self.Q = max(0.0, min(2000.0, 500.0 + self.tc_Kc * err_T + i_term))

        # LC
        if self.lc_mode == "Auto":
            err_L = self.Lvl - self.lc_stpt
            self.lc_err_integral += err_L * dt
            i_term = (self.lc_Kc / self.lc_tauI) * self.lc_err_integral if self.lc_tauI > 0 else 0
            self.v_out = max(0.0, min(10.0, 0.66 + self.lc_Kc * err_L + i_term))

        # 2. Physics & Dynamic Balance Approximations
        dT_dt = (self.Q - (self.T - self.T_in) * 2.0 - (self.T - 400.0) * 0.5) / 50.0
        self.T += dT_dt * dt

        vapor_gen = max(0.0, (self.T - 420.0) * 1.5)
        vapor_out = self.lift * (self.P / 10.0) * 2.0
        dP_dt = (vapor_gen - vapor_out) / 2.0
        self.P = max(10.0, self.P + dP_dt * dt)

        dLvl_dt = (self.F_in * (1.0 - self.x_butanol) * 0.1) - (self.v_out * 0.2)
        self.Lvl = max(0.0, min(100.0, self.Lvl + dLvl_dt * dt))

        # 3. Record History
        self.current_time += dt
        self.time_history.append(self.current_time)
        self.T_history.append(self.T)
        self.Q_history.append(self.Q)
        self.Lvl_history.append(self.Lvl)
        self.v_out_history.append(self.v_out)
        self.P_history.append(self.P)
        self.lift_history.append(self.lift)

        # Limit memory window (last 300 points)
        if len(self.time_history) > 300:
            self.time_history.pop(0)
            self.T_history.pop(0)
            self.Q_history.pop(0)
            self.Lvl_history.pop(0)
            self.v_out_history.pop(0)
            self.P_history.pop(0)
            self.lift_history.pop(0)


class FlashDrumGUI:
    def __init__(self, root):
        self.root = root
        self.root.title("Aniline-Butanol Flash Drum Simulator")
        self.root.geometry("1100x850")
        self.root.configure(bg="#e8e8e8")

        self.sim = FlashDrumSimulation()
        self.sim_speed = 2.0
        self.last_update_time = time.perf_counter()

        self.setup_ui()
        self.setup_plots()
        self.update_loop()

    def setup_ui(self):
        # Top Header Frame
        top_frame = tk.Frame(self.root, bg="#e8e8e8")
        top_frame.pack(side=tk.TOP, fill=tk.X, padx=10, pady=5)

        # Inlet Controls Frame
        inlet_box = tk.LabelFrame(top_frame, text="inlet conditions", font=("Arial", 11, "bold"), bg="white")
        inlet_box.pack(side=tk.LEFT, padx=10)

        tk.Button(inlet_box, text="Update", bg="#007bff", fg="white", font=("Arial", 9, "bold"),
                  command=self.update_inlets).grid(row=0, column=0, columnspan=2, pady=2, sticky="ew")

        tk.Label(inlet_box, text="F_in (mol/s)", bg="white").grid(row=1, column=0, sticky="w", padx=5)
        self.entry_Fin = tk.Entry(inlet_box, width=8)
        self.entry_Fin.insert(0, "15")
        self.entry_Fin.grid(row=1, column=1, padx=5, pady=2)

        tk.Label(inlet_box, text="x_butanol", bg="white").grid(row=2, column=0, sticky="w", padx=5)
        self.entry_xbut = tk.Entry(inlet_box, width=8)
        self.entry_xbut.insert(0, "0.5")
        self.entry_xbut.grid(row=2, column=1, padx=5, pady=2)

        tk.Label(inlet_box, text="T_in (K)", bg="white").grid(row=3, column=0, sticky="w", padx=5)
        self.entry_Tin = tk.Entry(inlet_box, width=8)
        self.entry_Tin.insert(0, "400")
        self.entry_Tin.grid(row=3, column=1, padx=5, pady=2)

        # Speed, Snapshot, Directions
        ctrl_box = tk.Frame(top_frame, bg="#e8e8e8")
        ctrl_box.pack(side=tk.LEFT, padx=20)

        speed_frame = tk.Frame(ctrl_box, bg="#d0e0e3", bd=1, relief="solid")
        speed_frame.pack(pady=2, fill=tk.X)
        self.speed_lbl = tk.Label(speed_frame, text=f"simulation speed: {int(self.sim_speed)}", bg="#d0e0e3", font=("Arial", 10))
        self.speed_lbl.pack()
        self.speed_slider = ttk.Scale(speed_frame, from_=1, to=10, value=2, command=self.change_speed)
        self.speed_slider.pack(padx=5, pady=2)

        tk.Button(ctrl_box, text="Snapshot .CSV", bg="#28a745", fg="white", font=("Arial", 10, "bold"),
                  command=self.save_csv).pack(pady=4, fill=tk.X)
        tk.Button(ctrl_box, text="Directions", bg="#5a6268", fg="white", font=("Arial", 10, "bold"),
                  command=self.show_directions).pack(pady=2, fill=tk.X)

        # Diagram & Controller Box Area
        center_frame = tk.Frame(self.root, bg="#e8e8e8")
        center_frame.pack(side=tk.TOP, fill=tk.BOTH, expand=True, padx=10, pady=5)

        # Left Controllers (TC)
        left_ctrl_frame = tk.Frame(center_frame, bg="#e8e8e8")
        left_ctrl_frame.pack(side=tk.LEFT, anchor="n", padx=10)

        self.tc_box = self.create_controller_ui(left_ctrl_frame, "temperature controller", "Q (kW)", "T stpt (K)",
                                                 "500", "455", "0", "36000", self.update_tc)

        # Center PFD Schematic (Canvas)
        self.canvas = tk.Canvas(center_frame, width=320, height=260, bg="#e8e8e8", highlightthickness=0)
        self.canvas.pack(side=tk.LEFT, expand=True)
        self.draw_pfd()

        # Right Controllers (PC and LC)
        right_ctrl_frame = tk.Frame(center_frame, bg="#e8e8e8")
        right_ctrl_frame.pack(side=tk.RIGHT, anchor="n", padx=10)

        self.pc_box = self.create_controller_ui(right_ctrl_frame, "pressure controller", "Lift", "P stpt (kPa)",
                                                 "0.5", "74.6", "0", "36000", self.update_pc)
        self.lc_box = self.create_controller_ui(right_ctrl_frame, "level controller", "v_out (L/s)", "Lvl stpt (%)",
                                                 "0.66", "17.6", "0", "36000", self.update_lc)

    def create_controller_ui(self, parent, title, mv_name, stpt_name, mv_val, stpt_val, kc_val, tau_val, update_cmd):
        box = tk.LabelFrame(parent, text=title, font=("Arial", 11, "bold"), bg="white")
        box.pack(pady=5, fill=tk.X)

        mode_frame = tk.Frame(box, bg="white")
        mode_frame.grid(row=0, column=0, columnspan=2, pady=2, sticky="ew")

        mode_var = tk.StringVar(value="Auto")
        auto_btn = tk.Radiobutton(mode_frame, text="Auto", value="Auto", variable=mode_var, indicatoron=False, width=6, bg="#6c757d", selectcolor="#007bff", fg="white")
        man_btn = tk.Radiobutton(mode_frame, text="Manual", value="Manual", variable=mode_var, indicatoron=False, width=6, bg="#6c757d", selectcolor="#007bff", fg="white")
        auto_btn.pack(side=tk.LEFT, padx=2)
        man_btn.pack(side=tk.LEFT, padx=2)

        tk.Button(mode_frame, text="Update", bg="#007bff", fg="white", font=("Arial", 8, "bold"), command=update_cmd).pack(side=tk.RIGHT, padx=2)

        labels = [mv_name, stpt_name, "Kc", "taul (s)"]
        defaults = [mv_val, stpt_val, kc_val, tau_val]
        entries = {}

        for idx, (lbl, df) in enumerate(zip(labels, defaults), start=1):
            tk.Label(box, text=lbl, bg="white", fg="blue" if idx==1 else ("red" if idx==2 else "black")).grid(row=idx, column=0, sticky="w", padx=5)
            e = tk.Entry(box, width=8)
            e.insert(0, df)
            e.grid(row=idx, column=1, padx=5, pady=2)
            entries[lbl] = e

        box.mode_var = mode_var
        box.entries = entries
        return box

    def setup_plots(self):
        plot_frame = tk.Frame(self.root, bg="white")
        plot_frame.pack(side=tk.BOTTOM, fill=tk.BOTH, expand=True, padx=10, pady=10)

        self.fig = Figure(figsize=(10, 3.0), dpi=100)
        
        # Subplot 1: Temp & Heat Duty
        self.ax1 = self.fig.add_subplot(131)
        self.ax1_twin = self.ax1.twinx()
        self.line_T, = self.ax1.plot([], [], 'g-', label="Temp (K)")
        self.line_Q, = self.ax1_twin.plot([], [], 'b-', label="Q (kW)")
        self.ax1.set_title("temperature and heat duty", fontsize=9)
        self.ax1.set_xlabel("time (s)", fontsize=8)
        self.ax1.set_ylabel("temperature (K)", color='g', fontsize=8)
        self.ax1_twin.set_ylabel("heat duty (kW)", color='b', fontsize=8)

        # Subplot 2: Level & Flow
        self.ax2 = self.fig.add_subplot(132)
        self.ax2_twin = self.ax2.twinx()
        self.line_Lvl, = self.ax2.plot([], [], 'g-', label="Level (%)")
        self.line_vout, = self.ax2_twin.plot([], [], 'b-', label="Flow (L/s)")
        self.ax2.set_title("level and bottoms flow rate", fontsize=9)
        self.ax2.set_xlabel("time (s)", fontsize=8)
        self.ax2.set_ylabel("level (%)", color='g', fontsize=8)
        self.ax2_twin.set_ylabel("bottoms flow (L / s)", color='b', fontsize=8)

        # Subplot 3: Pressure & Valve Lift
        self.ax3 = self.fig.add_subplot(133)
        self.ax3_twin = self.ax3.twinx()
        self.line_P, = self.ax3.plot([], [], 'g-', label="Pressure (kPa)")
        self.line_lift, = self.ax3_twin.plot([], [], 'b-', label="Lift")
        self.ax3.set_title("pressure and valve lift", fontsize=9)
        self.ax3.set_xlabel("time (s)", fontsize=8)
        self.ax3.set_ylabel("pressure (kPa)", color='g', fontsize=8)
        self.ax3_twin.set_ylabel("valve lift", color='b', fontsize=8)

        self.fig.tight_layout(pad=2.0)
        self.canvas_plot = FigureCanvasTkAgg(self.fig, master=plot_frame)
        self.canvas_plot.get_tk_widget().pack(fill=tk.BOTH, expand=True)

    def draw_pfd(self):
        self.canvas.delete("all")
        # Drum body
        self.canvas.create_rectangle(120, 50, 200, 200, fill="#d3d3d3", outline="black", width=2)
        self.canvas.create_arc(120, 30, 200, 70, start=0, extent=180, fill="#d3d3d3", outline="black", width=2)
        self.canvas.create_arc(120, 180, 200, 220, start=180, extent=180, fill="#d3d3d3", outline="black", width=2)

        # Liquid level in drum
        lvl_h = max(5, int((self.sim.Lvl / 100.0) * 120))
        self.canvas.create_rectangle(121, 200 - lvl_h, 199, 200, fill="#d2b48c", outline="")

        # Label on drum
        self.canvas.create_text(160, 90, text="aniline\nbutanol", font=("Arial", 10, "bold"))

        # Controller badges & dynamic numbers
        self.canvas.create_oval(230, 30, 270, 70, fill="white", outline="black", width=2)
        self.canvas.create_text(250, 50, text="PC", font=("Arial", 11, "bold"))
        self.canvas.create_text(250, 18, text=f"{self.sim.P:.1f} kPa", fill="green", font=("Arial", 10, "bold"))

        self.canvas.create_oval(230, 140, 270, 180, fill="white", outline="black", width=2)
        self.canvas.create_text(250, 160, text="LC", font=("Arial", 11, "bold"))
        self.canvas.create_text(250, 122, text=f"{self.sim.Lvl:.1f} %", fill="green", font=("Arial", 10, "bold"))

        self.canvas.create_oval(50, 140, 90, 180, fill="white", outline="black", width=2)
        self.canvas.create_text(70, 160, text="TC", font=("Arial", 11, "bold"))
        self.canvas.create_text(70, 198, text=f"{self.sim.T:.0f} K", fill="green", font=("Arial", 10, "bold"))

    def update_inlets(self):
        try:
            self.sim.F_in = float(self.entry_Fin.get())
            self.sim.x_butanol = float(self.entry_xbut.get())
            self.sim.T_in = float(self.entry_Tin.get())
        except ValueError:
            messagebox.showerror("Error", "Invalid numeric inlet conditions")

    def update_pc(self):
        try:
            self.sim.pc_mode = self.pc_box.mode_var.get()
            self.sim.lift = float(self.pc_box.entries["Lift"].get())
            self.sim.pc_stpt = float(self.pc_box.entries["P stpt (kPa)"].get())
            self.sim.pc_Kc = float(self.pc_box.entries["Kc"].get())
            self.sim.pc_tauI = float(self.pc_box.entries["taul (s)"].get())
        except ValueError:
            messagebox.showerror("Error", "Invalid input in Pressure Controller")

    def update_tc(self):
        try:
            self.sim.tc_mode = self.tc_box.mode_var.get()
            self.sim.Q = float(self.tc_box.entries["Q (kW)"].get())
            self.sim.tc_stpt = float(self.tc_box.entries["T stpt (K)"].get())
            self.sim.tc_Kc = float(self.tc_box.entries["Kc"].get())
            self.sim.tc_tauI = float(self.tc_box.entries["taul (s)"].get())
        except ValueError:
            messagebox.showerror("Error", "Invalid input in Temperature Controller")

    def update_lc(self):
        try:
            self.sim.lc_mode = self.lc_box.mode_var.get()
            self.sim.v_out = float(self.lc_box.entries["v_out (L/s)"].get())
            self.sim.lc_stpt = float(self.lc_box.entries["Lvl stpt (%)"].get())
            self.sim.lc_Kc = float(self.lc_box.entries["Kc"].get())
            self.sim.lc_tauI = float(self.lc_box.entries["taul (s)"].get())
        except ValueError:
            messagebox.showerror("Error", "Invalid input in Level Controller")

    def change_speed(self, val):
        self.sim_speed = float(val)
        self.speed_lbl.config(text=f"simulation speed: {int(self.sim_speed)}")

    def show_directions(self):
        messagebox.showinfo("Directions", "Flash Drum Control System Simulation\n\n"
                                          "1. Set inlet conditions on top-left.\n"
                                          "2. Adjust control parameters (Auto/Manual, Setpoint, Kc, TauI) for TC, PC, and LC.\n"
                                          "3. Click 'Update' on the respective frame to apply changes.\n"
                                          "4. Export live data using 'Snapshot .CSV'.")

    def save_csv(self):
        file_path = filedialog.asksaveasfilename(defaultextension=".csv", filetypes=[("CSV Files", "*.csv")])
        if file_path:
            with open(file_path, "w", newline="") as f:
                writer = csv.writer(f)
                writer.writerow(["Time (s)", "Temperature (K)", "Heat Duty (kW)", "Level (%)", "Bottoms Flow (L/s)", "Pressure (kPa)", "Valve Lift"])
                for t, T, Q, Lvl, v, P, lift in zip(self.sim.time_history, self.sim.T_history, self.sim.Q_history,
                                                    self.sim.Lvl_history, self.sim.v_out_history, self.sim.P_history, self.sim.lift_history):
                    writer.writerow([t, T, Q, Lvl, v, P, lift])
            messagebox.showinfo("Success", "Data saved successfully.")

    def update_loop(self):
        current_clock = time.perf_counter()
        elapsed_real_seconds = current_clock - self.last_update_time
        self.last_update_time = current_clock

        # Calculate time step (dt) scaled relative to real clock time and speed multiplier
        dt = elapsed_real_seconds * self.sim_speed

        # Advance physics engine using exact elapsed dt
        self.sim.step(dt=dt)

        # Update PFD drawing
        self.draw_pfd()

        # Update Plot Lines Efficiently
        if self.sim.time_history:
            times = [t - self.sim.current_time for t in self.sim.time_history]

            # Subplot 1
            self.line_T.set_data(times, self.sim.T_history)
            self.line_Q.set_data(times, self.sim.Q_history)
            self.ax1.relim(); self.ax1.autoscale_view()
            self.ax1_twin.relim(); self.ax1_twin.autoscale_view()

            # Subplot 2
            self.line_Lvl.set_data(times, self.sim.Lvl_history)
            self.line_vout.set_data(times, self.sim.v_out_history)
            self.ax2.relim(); self.ax2.autoscale_view()
            self.ax2_twin.relim(); self.ax2_twin.autoscale_view()

            # Subplot 3
            self.line_P.set_data(times, self.sim.P_history)
            self.line_lift.set_data(times, self.sim.lift_history)
            self.ax3.relim(); self.ax3.autoscale_view()
            self.ax3_twin.relim(); self.ax3_twin.autoscale_view()

            self.canvas_plot.draw_idle()

        # Re-trigger UI frame loop after ~50ms (~20 FPS update rate)
        self.root.after(50, self.update_loop)

if __name__ == "__main__":
    root = tk.Tk()
    app = FlashDrumGUI(root)
    root.mainloop()
Key Changes Implemented
Real-time Delta (dt) Tracking: Replaced the fixed step count loop with precise time sampling (time.perf_counter()).

Hardware Independent: The simulation clock advances based on actual wall-clock seconds scaled by sim_speed, guaranteeing consistent integration speed regardless of CPU performance